In [50]:
import os
import pandas as pd

caminho_pasta = 'C:/Users/A0050637/Downloads/2025/'
lista_colunas = [f'Coluna_{i}' for i in range(26)]
dataframes_processados = []

# Configurações globais de exibição do Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# Verifica se a pasta existe antes de iniciar
if not os.path.exists(caminho_pasta):
    print(f"Erro: A pasta '{caminho_pasta}' não existe.")
else:
    arquivos = os.listdir(caminho_pasta)
    print(f"Total de arquivos encontrados na pasta: {len(arquivos)}")

    for nome_arquivo in arquivos:
        if nome_arquivo.lower().endswith('.csv'):
            caminho_completo = os.path.join(caminho_pasta, nome_arquivo)
            print(f"Processando: {nome_arquivo}")
            
            try:
                df = pd.read_csv(
                    caminho_completo, 
                    sep=';', 
                    encoding='latin-1',
                    names=lista_colunas, 
                    header=None
                )
                
                REGIAO = df.iloc[0, 1]
                UF = df.iloc[1, 1]
                ESTACAO = df.iloc[2, 1]
                COD = df.iloc[3, 1]       
                LATITUDE = df.iloc[4, 1]
                LONGITUDE = df.iloc[5, 1]
                ALTITUDE = df.iloc[6, 1]
                DATA_FUNDACAO = df.iloc[7, 1] 
                
                df = df.iloc[9:].copy()
                
                df["REGIAO"] = REGIAO
                df["UF"] = UF
                df["ESTACAO"] = ESTACAO
                df["COD"] = COD
                df["LATITUDE"] = LATITUDE
                df["LONGITUDE"] = LONGITUDE
                df["ALTITUDE"] = ALTITUDE
                df["DATA_FUNDACAO"] = DATA_FUNDACAO
                
                df = df.rename(columns={
                    'Coluna_0': 'DATA', 'Coluna_1': 'HORA', 'Coluna_2': 'PRECIPITACAO_mm',
                    'Coluna_3': 'PRESSAO_ATM_ESTACAO_mB', 'Coluna_4': 'PRESSAO_ATM_MAX_AUT_mB',
                    'Coluna_5': 'PRESSAO_ATM_MIN_AUT_mB', 'Coluna_6': 'RADIACAO_kJM2',
                    'Coluna_7': 'TEMPERATURA_C', 'Coluna_8': 'TEMPERATURA_P_ORVALHO_C',
                    'Coluna_9': 'TEMPERATURA_MAX_C', 'Coluna_10': 'TEMPERATURA_MIN_C',
                    'Coluna_11': 'TEMPERATURA_ORVALHO_MAX_C', 'Coluna_12': 'TEMPERATURA_ORVALHO_MIN_C',
                    'Coluna_13': 'UMIDADE_MAX', 'Coluna_14': 'UMIDADE_MIN', 'Coluna_15': 'UMIDADE_RELATIVA',
                    'Coluna_16': 'VENTO_HORARIO', 'Coluna_17': 'VENTO_RAJADA_MAX', 'Coluna_18': 'VENTO_VELOCIDADE'
                })
                
                df_final = df[[
                    'DATA', 'HORA', 'PRECIPITACAO_mm', 'PRESSAO_ATM_ESTACAO_mB', 'PRESSAO_ATM_MAX_AUT_mB', 
                    'PRESSAO_ATM_MIN_AUT_mB', 'RADIACAO_kJM2', 'TEMPERATURA_C', 'TEMPERATURA_P_ORVALHO_C', 
                    'TEMPERATURA_MAX_C', 'TEMPERATURA_MIN_C', 'TEMPERATURA_ORVALHO_MAX_C', 'TEMPERATURA_ORVALHO_MIN_C', 
                    'UMIDADE_MAX', 'UMIDADE_MIN', 'UMIDADE_RELATIVA', 'VENTO_HORARIO', 'VENTO_RAJADA_MAX', 
                    'VENTO_VELOCIDADE', 'REGIAO', 'UF', 'ESTACAO', 'COD', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'DATA_FUNDACAO'
                ]].copy()
                
                dataframes_processados.append(df_final)
                
            except Exception as e:
                print(f"Erro ao processar o arquivo {nome_arquivo}: {e}")

    # ================= CORREÇÃO DO ERRO AQUI =================
    if len(dataframes_processados) > 0:
        # 1. Consolida a lista de tabelas em um único DataFrame chamado df_consolidado
        df_consolidado = pd.concat(dataframes_processados, ignore_index=True)
        
        # 2. Converte o texto da coluna DATA para o formato datetime do Pandas (Padrão INMET: AAAA-MM-DD)
        df_consolidado['DATA'] = pd.to_datetime(df_consolidado['DATA'], errors='coerce')
        
        # 3. Agora sim, extrai e cria a coluna ANO_MES no formato AA_mm
        df_consolidado['ANO_MES'] = df_consolidado['DATA'].dt.strftime('%y_%m')

    else:
        print("\nNenhum arquivo CSV válido pôde ser processado.")




Total de arquivos encontrados na pasta: 3
Processando: INMET_S_RS_B839_SANTO ANTONIO DA PATRULHA_13-10-2025_A_31-12-2025.CSV
Processando: INMET_S_RS_B841_TORRES_13-10-2025_A_31-12-2025.CSV
Processando: teste_a.CSV


In [51]:
# 1. Lista das colunas que precisam virar números para o cálculo
colunas_numericas = [
    'PRECIPITACAO_mm', 'TEMPERATURA_C', 'TEMPERATURA_MAX_C', 'TEMPERATURA_MIN_C',
    'UMIDADE_MAX', 'UMIDADE_MIN', 'UMIDADE_RELATIVA', 
    'VENTO_HORARIO', 'VENTO_RAJADA_MAX', 'VENTO_VELOCIDADE'
]

# 2. Loop para converter vírgula em ponto e transformar em número (float)
for col in colunas_numericas:
    # Garante que a coluna seja tratada como texto temporariamente para a substituição
    df_consolidado[col] = df_consolidado[col].astype(str)
    # Substitui a vírgula decimal brasileira pelo ponto decimal americano
    df_consolidado[col] = df_consolidado[col].str.replace(',', '.', regex=False)
    # Transforma em número. O errors='coerce' transforma textos inválidos (ex: '-', 'null') em NaN (vazio)
    df_consolidado[col] = pd.to_numeric(df_consolidado[col], errors='coerce')


# ================= AGORA O SEU AGRUPAMENTO VAI FUNCIONAR =================

colunas_agrupamento = [
    'REGIAO', 'UF', 'ESTACAO', 'COD', 
    'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'DATA_FUNDACAO', 'ANO_MES'
]

df_resumo = df_consolidado.groupby(colunas_agrupamento).agg({
    'PRECIPITACAO_mm': 'sum',
    'TEMPERATURA_C': 'mean',
    'TEMPERATURA_MAX_C': 'mean',
    'TEMPERATURA_MIN_C': 'mean',
    'UMIDADE_MAX': 'mean',
    'UMIDADE_MIN': 'mean',
    'UMIDADE_RELATIVA': 'mean',
    'VENTO_HORARIO': 'mean',
    'VENTO_RAJADA_MAX': 'mean',
    'VENTO_VELOCIDADE': 'mean'
}).reset_index()

print("=== SUCESSO: DataFrame 'df_resumo' criado sem erros! ===")
print(df_resumo.head(1000))



=== SUCESSO: DataFrame 'df_resumo' criado sem erros! ===
   REGIAO  UF                      ESTACAO   COD      LATITUDE     LONGITUDE ALTITUDE DATA_FUNDACAO ANO_MES  PRECIPITACAO_mm  TEMPERATURA_C  TEMPERATURA_MAX_C  TEMPERATURA_MIN_C  UMIDADE_MAX  UMIDADE_MIN  UMIDADE_RELATIVA  VENTO_HORARIO  VENTO_RAJADA_MAX  VENTO_VELOCIDADE
0       S  RS    SANTO ANTONIO DA PATRULHA  B839  -29,81611111  -50,53333333       47      13/10/25   25_10             61.4      19.391640          19.889711          18.904502    75.070740    68.810289         71.958199     128.421222          5.089068          2.767203
1       S  RS    SANTO ANTONIO DA PATRULHA  B839  -29,81611111  -50,53333333       47      13/10/25   25_11             92.0      21.114123          21.625250          20.615121    77.259629    71.215407         74.168331     153.681883          4.624679          2.539658
2       S  RS    SANTO ANTONIO DA PATRULHA  B839  -29,81611111  -50,53333333       47      13/10/25   25_12            151.8